# ZETRO Final Notebook

## Overview

This notebook contains the final training and submission pipeline for the Myllia Cell Competition. The model learns perturbation-response deltas over 5,127 genes using a fold-local embedding pipeline, a low-capacity hypernetwork, and augmentation from single-cell h5ad data. Augmentation improved public leaderboard performance even though there was barely any coverage on the test perts. I took a risk and hoped that the private perts were similar to the public perts. That risk did not pay off and I dropped from 5th place to 9th place. These markdown cells were generated using AI and then fully edited and fixed by me.

## Notebook structure

1. Import libraries and define global settings
2. Load training tables and construct delta targets
3. Define model and augmentation hyperparameters
4. Cache normalized single-cell control data from `training_cells.h5ad`
5. Build fold-local artifacts for perturbation embeddings and response basis initialization
6. Define the loss and official scoring logic
7. Run grouped cross-validation and tune global shrinkage
8. Refit on all training perturbations
9. Predict all released perturbations and write the final submission

## Inputs

- `data/training_data_means.csv`
- `data/training_data_ground_truth_table.csv`
- `data/pert_ids_val.csv`
- `data/pert_ids_all.csv`
- `data/sample_submission.csv`
- `data/training_cells.h5ad`

## Outputs

- grouped CV diagnostics
- tuned `ALPHA_SHRINK`
- `MEDIAN_EPOCHS`
- final submission CSV

## 1. Imports and global settings

This cell imports the required libraries and defines the global notebook configuration.

The main training settings here control:
- optimization behavior
- evaluation frequency
- model seeds
- gradient clipping
- project root paths

These values are shared across both cross-validation and the final refit stage.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

device = device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LR = 2.0e-4
WD = 1e-4 * 2
EPOCHS = 400
BATCH_GENES = 32 # minibatch over perturbed genes
EVAL_EVERY = 5
PATIENCE = 10 # early stopping patience in eval steps

GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

MODEL_SEEDS = [90, 70, 80]
GRAD_CLIP = 1.0

H5AD_PATH = ROOT / "data" / "training_cells.h5ad"
PERT_IDS_ALL_CSV = ROOT / "data" / "pert_ids_all.csv"

## 2. Load competition tables and build training deltas

This cell loads the mean-expression training table, validation mapping table, and sample submission.

The non-targeting row is used as the baseline reference. Every perturbation is converted into a delta vector. This gives the main training target matrix `D_train` with shape `(80, 5127)`.

In [ ]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

D_train = df_train[gene_columns].to_numpy(np.float32) - x_base[None, :]
delta_baseline = D_train.mean(axis=0).astype(np.float32)

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping rows:", len(df_valmap))

Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


## 3. Hyperparameters

The hyperparameters are grouped into:
- `h5ad` augmentation parameters
- control-based embedding settings
- model dimensions
- basis regularization
- SVD embedding behavior
- post-prediction shrinkage search grid

These values determine both the representation pipeline and the final model capacity.

In [ ]:
# h5ad augmentation
AUGMENT_H5AD = True
AUG_WARMUP_EPOCHS = 10
AUG_RAMP_EPOCHS = 20
AUG_P_MAX = 1.0

BOOT_K = 32
BOOT_M = 256
BOOT_SEED = SEED

AUG_SLOPE_CLAMP_MIN = 0.2
AUG_SLOPE_CLAMP_MAX = 5.0
AUG_MIN_FIT_PERTS = 8

H5AD_TOPK = 256

# model sizes
EMB_DIM_PERT = 32
BASIS_K = 32
HNET_HIDDEN = 128
HNET_DROPOUT = 0.15

# basis regularization
LAMBDA_BASIS_DRIFT = 5e-5
LAMBDA_BASIS_GAIN = 1e-4

# embedding behavior
SVD_SIGMA_POWER = 0.5
SVD_ROW_NORM = True

ALPHA_GRID = np.linspace(0.3, 1.1, 100).astype(np.float32).tolist()

## 4. Single-cell cache and control-derived perturbation embeddings

This section prepares features from `training_cells.h5ad`.

The main purpose is to build perturbation embeddings for genes using control-cell geometry. The notebook:
- normalizes single-cell counts to CPM10K
- applies log transform
- identifies non-targeting control cells
- standardizes control-cell expression over the 5,127 output genes
- projects control-derived gene correlations into the fold-local perturbation embedding space

This lets the model build perturbation embeddings even for genes that are not directly represented by the competition training rows. I did a lot of experimentation with `training_cells.h5ad`. It improved public leaderboard score, but seemed to lower private leaderboard score. I may do more experimentation when I get home from my trip because I want to find the reason behind the private leaderboard drop.

In [ ]:
_H5AD_CACHE = None

def load_h5ad_ctrl_cache(h5ad_path: Path, gene_columns):
    """
    Cache:
      - Xn (CPM10K + log2(1+x)) sparse
      - ctrl mask
      - var map (GENE->idx)
      - out_idx mapping for the output gene columns
      - Xout_z (control cells, standardized per output gene)
      - obs_pertU
    """
    global _H5AD_CACHE
    if _H5AD_CACHE is not None:
        return _H5AD_CACHE

    adata = ad.read_h5ad(str(h5ad_path))
    pert_col = "sgrna_symbol"

    Xc = adata.X
    Xc = Xc.tocsr()

    # CPM10K normalize, then log2(1+x)
    cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
    scale = (10000.0 / np.clip(cell_sum, 1e-12, None)).astype(np.float64)
    Xn = Xc.multiply(scale[:, None]).tocsr()
    Xn.data = np.log1p(Xn.data) / np.log(2.0)

    obs_pert = adata.obs[pert_col].astype(str).to_numpy()
    obs_pertU = np.char.upper(obs_pert.astype("U"))

    ctrl_mask = (obs_pertU == "NON-TARGETING")

    var_names = adata.var_names.astype(str).to_numpy()
    varU = np.char.upper(var_names.astype("U"))
    var = {varU[i]: i for i in range(len(varU))}

    out_idx = np.array([var[str(g).upper()] for g in gene_columns], dtype=np.int64)

    Xn_ctrl = Xn[ctrl_mask]
    Xout = Xn_ctrl[:, out_idx]
    Xout = Xout.toarray()
    Xout = Xout.astype(np.float32)

    # z-score per output gene over control cells
    mu = Xout.mean(axis=0, keepdims=True)
    sd = Xout.std(axis=0, keepdims=True) + 1e-6
    Xout_z = ((Xout - mu) / sd).astype(np.float32)

    _H5AD_CACHE = dict(
        Xn=Xn,
        Xn_ctrl=Xn_ctrl,
        ctrl_mask=ctrl_mask,
        var=var,
        out_idx=out_idx,
        Xout_z=Xout_z,
        obs_pertU=obs_pertU,
    )
    print("[h5ad] cache built:",
          "n_cells=", Xn.shape[0],
          "n_genes=", Xn.shape[1],
          "n_ctrl=", int(ctrl_mask.sum()))
    return _H5AD_CACHE

def build_z_from_ctrl_corr(cache, genesU, P_out, topk=256):
    """
    Build a Z embedding for each gene in genesU using correlation of that gene's
    control-cell expression with standardized output genes (Xout_z), projected
    into the SVD basis P_out (G x d_pert).

    Returns dict: {geneU: z (d_pert,)}
    """
    Xout_z = cache["Xout_z"]      # (n_ctrl, G_out) standardized
    Xn_ctrl = cache["Xn_ctrl"]    # (n_ctrl, G_all) CPM/log space
    var = cache["var"]

    out = {}
    for gU in genesU:
        j = var.get(str(gU).upper(), None)
        if j is None:
            continue

        xg = Xn_ctrl[:, j]
        xg = xg.toarray()
        xg = np.asarray(xg).ravel().astype(np.float32)
        xg = (xg - xg.mean()) / (xg.std() + 1e-6)

        corr = (xg[:, None] * Xout_z).mean(axis=0)  # (G_out,)
        idx = np.argsort(-np.abs(corr))[:int(topk)]
        w = corr[idx].astype(np.float32)

        z = (w[:, None] * P_out[idx]).sum(axis=0)   # (d_pert,)
        z = z / (np.linalg.norm(z) + 1e-12)
        out[str(gU).upper()] = z.astype(np.float32)

    return out

## 5. Fold-local artifact construction

For each cross-validation fold, the notebook rebuilds the representation pipeline using only the training perturbations in that fold.

This step:
- applies a signed-log transform to the training deltas
- fits fold-local SVD embeddings for perturbation representation
- builds control-correlation perturbation embeddings from `training_cells.h5ad`
- initializes the response basis used by the hypernetwork
- computes the gene-bias initialization from the fold training mean

This keeps the representation fold-local and avoids leaking validation perturbations into the embedding space.

In [ ]:
def _row_l2_normalize(x, eps=1e-12):
    x = np.asarray(x, np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return (x / np.clip(n, eps, None)).astype(np.float32)

def _vec_l2_normalize(x, eps=1e-12):
    x = np.asarray(x, np.float32)
    n = np.linalg.norm(x)
    return (x / max(float(n), eps)).astype(np.float32)

def build_fold_artifacts(tr_idx, query_genes):
    tr_idx = np.asarray(tr_idx, dtype=np.int64)
    query_genes = [str(g).strip().upper() for g in query_genes]
    tr_genes = [str(train_genes[i]).strip().upper() for i in tr_idx]

    Ytr = D_train[tr_idx].astype(np.float32, copy=True)
    Xtr = np.sign(Ytr) * np.log2(1.0 + np.abs(Ytr))

    k_svd = min(int(EMB_DIM_PERT), int(Xtr.shape[0] - 1), int(Xtr.shape[1] - 1))
    svd = TruncatedSVD(n_components=k_svd, random_state=SEED)
    svd.fit(Xtr)

    V = svd.components_.T.astype(np.float32)
    S = svd.singular_values_.astype(np.float32)

    scale = np.power(np.clip(S[:k_svd], 1e-8, None), float(SVD_SIGMA_POWER)).astype(np.float32)
    gene_emb_pert_all = (V[:, :k_svd] * scale[None, :]).astype(np.float32)

    if SVD_ROW_NORM:
        gene_emb_pert_all = _row_l2_normalize(gene_emb_pert_all)

    gene2emb_pert_svd = {
        str(gene_columns[i]).upper(): gene_emb_pert_all[i].copy()
        for i in range(len(gene_columns))
    }
    emb_fallback_pert = _vec_l2_normalize(gene_emb_pert_all.mean(axis=0))

    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)
    needed_allU = sorted(set(tr_genes + query_genes))
    ctrl_pert_all = build_z_from_ctrl_corr(
        cache,
        needed_allU,
        gene_emb_pert_all.astype(np.float32),
        topk=int(H5AD_TOPK),
    )

    if SVD_ROW_NORM:
        ctrl_pert_all = {
            gU: _vec_l2_normalize(z).astype(np.float32)
            for gU, z in ctrl_pert_all.items()
        }

    def emb_pert_fold(g):
        gU = str(g).strip().upper()
        if gU in ctrl_pert_all:
            return ctrl_pert_all[gU].astype(np.float32)
        if gU in gene2emb_pert_svd:
            return gene2emb_pert_svd[gU].astype(np.float32)
        return emb_fallback_pert.astype(np.float32)

    Z_ref = np.vstack([emb_pert_fold(g) for g in tr_genes]).astype(np.float32)
    Z_query = np.vstack([emb_pert_fold(g) for g in query_genes]).astype(np.float32)

    basis_k_eff = min(int(BASIS_K), int(Ytr.shape[0] - 1), int(Ytr.shape[1] - 1))

    Y_center = Ytr - Ytr.mean(axis=0, keepdims=True)
    svd_basis = TruncatedSVD(n_components=basis_k_eff, random_state=SEED)
    svd_basis.fit(Y_center)

    BASIS_INIT = svd_basis.components_.T.astype(np.float32)
    BASIS_INIT = BASIS_INIT / np.clip(
        np.linalg.norm(BASIS_INIT, axis=0, keepdims=True), 1e-12, None
    ).astype(np.float32)

    GENE_BIAS_INIT = Ytr.mean(axis=0).astype(np.float32)

    ctrl_ref_cov = sum(g in ctrl_pert_all for g in tr_genes)
    ctrl_query_cov = sum(g in ctrl_pert_all for g in query_genes)

    return {
        "query_genes": query_genes,
        "Y_ref": Ytr,
        "Z_ref": Z_ref,
        "Z_query": Z_query,
        "BASIS_INIT": BASIS_INIT,
        "GENE_BIAS_INIT": GENE_BIAS_INIT,
        "ctrl_ref_cov": int(ctrl_ref_cov),
        "ctrl_query_cov": int(ctrl_query_cov),
        "svd_var_sum": float(svd.explained_variance_ratio_.sum()),
        "basis_var_sum": float(svd_basis.explained_variance_ratio_.sum()),
        "d_pert": int(k_svd),
        "basis_k_eff": int(basis_k_eff),
    }

## 6. Loss functions

This section defines the training loss used to fit the model.

The core loss is a weighted L1-like objective that emphasizes genes with larger true effects. A smooth gate is applied to the magnitude of the true delta so that the model focuses more strongly on biologically meaningful shifts and less on near-zero noise.

A second row-weighting step uses the competition baseline difficulty so harder rows do not dominate the optimization.

Squaring the gate improved performance, but was prone to overfitting.

Heavily debated between w, w\*\*1.5 and w\*\*2. From the tiny bit of information I gathered due to the limited amount of submissions w\*\*2 overfitted and caused LB performance to drop. w\*\*1.5 had a higher CV score and LB score than the other two.

In [ ]:
def gate_smoothstep(x, a=GATE_A, b=GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(
    delta_true: torch.Tensor,   # (N, G)
    delta_pred: torch.Tensor,   # (N, G)
    eps: float = EPS
) -> torch.Tensor:
    """
    Per-row weighted L1-like using w**1.5 on gate(|delta_true|).
    Returns: (N,) per-row losses.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    w2 = w ** 1.5

    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w2 * err, dim=1)                                # (N,)
    den = torch.clamp(torch.sum(w2, dim=1), min=eps)                # (N,)
    return num / den                                                # (N,)

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Row-weighted version of per_row_weighted_l1_like.

    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)
    w = 1.0 / torch.sqrt(b + eps)

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

## 7. Hypernetwork model

This notebook uses a compact basis hypernetwork.

The model takes a perturbation embedding as input, passes it through a 2-layer MLP, and predicts coefficients over a learned response basis. The final output is reconstructed as:

yhat = c(z) * B.T + b_gene + b_global

where:
- `z` is the perturbation embedding
- `c(z)` is the coefficient vector predicted by the MLP
- `B` is the learned response basis
- `b_gene` and `b_global` are learned bias terms

This keeps the model small while still letting it express structured gene-response patterns.

In [ ]:
class PlainBasisHyperNet(nn.Module):
    def __init__(self, d_pert, basis_init, hidden, dropout):
        super().__init__()

        G, K = basis_init.shape
        self.G = G
        self.K = K

        self.register_buffer("basis0", basis_init.detach().clone())

        self.coef_net = nn.Sequential(
            nn.Linear(d_pert, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, K),
        )

        self.basis_delta = nn.Parameter(torch.zeros_like(self.basis0))
        self.basis_gain = nn.Parameter(torch.ones(K, device=basis_init.device))
        self.bias_gene = nn.Parameter(torch.zeros(G, device=basis_init.device))
        self.bias_global = nn.Parameter(torch.zeros(1, device=basis_init.device))

    def set_gene_bias_init(self, bias_init):
        with torch.no_grad():
            self.bias_gene.copy_(bias_init)

    def get_basis(self):
        return self.basis0 * self.basis_gain[None, :] + self.basis_delta

    def forward_parts(self, z_pert):
        c = self.coef_net(z_pert)
        B = self.get_basis()
        y = c @ B.T
        y = y + self.bias_gene[None, :] + self.bias_global
        return y, c

    def forward(self, z_pert):
        return self.forward_parts(z_pert)[0]

## 8. Competition weights and baseline difficulty

The official competition metric uses per-gene weights and a baseline row difficulty term. This cell loads those quantities from the ground-truth table and aligns them with the 80 training perturbations.

These values are used later for:
- the official validation score
- row reweighting during training

In [ ]:
gt_path = ROOT / "data" / "training_data_ground_truth_table.csv"
gt_df = pd.read_csv(gt_path)

sol_aligned = gt_df.set_index("pert_id").loc[train_genes].reset_index()

w_cols = [f"w_{g}" for g in gene_columns]
sol_aligned = sol_aligned[["pert_id"] + list(gene_columns) + w_cols + ["baseline_wmae"]]

baseline_wmae = sol_aligned["baseline_wmae"].to_numpy(np.float32)
baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

W_gene = sol_aligned[w_cols].to_numpy(np.float32)

W64 = W_gene.astype(np.float64)
row_sums = W64.sum(axis=1, keepdims=True)
W64 *= (W64.shape[1] / np.maximum(row_sums, 1e-300))
W64[:, -1] += (W64.shape[1] - W64.sum(axis=1))
W_gene = W64.astype(np.float32)

## 9. Official scoring function

This section implements the competition score in delta space.

The score combines two pieces:
1. a weighted error improvement term relative to the baseline
2. a positive weighted cosine similarity term

The product of these two components gives the final score used for model selection and alpha tuning.

In [ ]:
def score_delta(dt, dp, idx):
    dt = np.asarray(dt, np.float64)
    dp = np.asarray(dp, np.float64)
    w = W_gene[idx].astype(np.float64, copy=False)
    base = baseline_wmae[idx].astype(np.float64, copy=False)

    abs_err = np.abs(dt - dp)
    pred_wmae = np.mean(abs_err * w, axis=1)
    pred_wmae = np.maximum(pred_wmae, 1e-12)
    base = np.maximum(base, 1e-12)

    terms = np.log2(base / pred_wmae)
    terms = np.minimum(terms, 5.0)
    sum_wmae = float(np.sum(terms))
    mean_term = float(np.mean(terms))

    a = dp.ravel()
    b = dt.ravel()
    x = np.maximum(np.abs(a), np.abs(b))
    t = np.clip(x / 0.2, 0.0, 1.0)
    w_gate = t * t * (3.0 - 2.0 * t)
    w2 = w_gate * w_gate

    num = np.sum(w2 * a * b)
    den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
    wcos = 0.0 if den < 1e-12 else float(num / den)
    wcos_pos = max(0.0, wcos)

    raw = sum_wmae * wcos_pos
    score = round(raw, 5)

    # normalized (scale-free) metric
    score_per_row = mean_term * wcos_pos

    return {
        "score": score,
        "raw": float(raw),
        "sum_wmae": sum_wmae,
        "mean_term": mean_term,
        "wcos": wcos,
        "score_per_row": float(score_per_row),
        "n_rows": int(len(idx)),
    }

## 10. Precompute `h5ad` bootstrap augmentation targets

This cell builds augmentation targets from the single-cell dataset.

For each training perturbation with matching single-cell rows, the notebook computes:
- the mean delta in `h5ad` space
- multiple bootstrap resamples of that perturbation

Later, during training, these `h5ad` deltas are affinely mapped into competition space and mixed into the batch targets after a warmup period.

In [ ]:
# H5ad data
boot_raw = None
h5_mean_raw = None
boot_ok = None

if AUGMENT_H5AD:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)

    Xn = cache["Xn"]               # sparse (n_cells, 19226) normalized CPM10K + log2
    out_idx = cache["out_idx"]     # (5127,)
    obs_pertU = cache["obs_pertU"] # (n_cells,)
    ctrl_mask = cache["ctrl_mask"]

    # control mean over output genes (in same h5ad normalized space)
    ctrl_mean = Xn[ctrl_mask][:, out_idx].mean(axis=0)
    ctrl_mean = ctrl_mean.A
    ctrl_mean = np.asarray(ctrl_mean).ravel().astype(np.float32)  # (5127,)

    N = len(train_genes)
    G = len(gene_columns)

    h5_mean_raw = np.zeros((N, G), dtype=np.float32)
    boot_raw = np.zeros((N, BOOT_K, G), dtype=np.float32)
    boot_ok = np.zeros((N,), dtype=np.int32)

    rng = np.random.RandomState(BOOT_SEED)

    for i, g in enumerate(train_genes.tolist()):
        gU = str(g).upper()
        rows = np.where(obs_pertU == gU)[0]
        if len(rows) == 0:
            continue

        boot_ok[i] = 1

        # mean delta using all cells for this pert
        Xi = Xn[rows][:, out_idx].mean(axis=0)
        Xi = Xi.A
        Xi = np.asarray(Xi).ravel().astype(np.float32)
        h5_mean_raw[i] = (Xi - ctrl_mean)

        # bootstraps
        for k in range(BOOT_K):
            samp = rng.choice(rows, size=min(int(BOOT_M), len(rows)), replace=True)
            Xk = Xn[samp][:, out_idx].mean(axis=0)
            Xk = Xk.A
            Xk = np.asarray(Xk).ravel().astype(np.float32)
            boot_raw[i, k] = (Xk - ctrl_mean)

    print("[aug] boot_ok:", int(boot_ok.sum()), "/", N)
    print("[aug] h5_mean_raw:", h5_mean_raw.shape, "boot_raw:", boot_raw.shape)

[h5ad] cache built: n_cells= 17882 n_genes= 19226 n_ctrl= 1026
[aug] boot_ok: 80 / 80
[aug] h5_mean_raw: (80, 5127) boot_raw: (80, 32, 5127)


## 11. Grouped cross-validation, early stopping, and shrinkage tuning

This is the main training loop.

For each grouped fold:
- fold-local artifacts are rebuilt
- one model is trained per seed
- validation predictions are averaged across seeds
- shrinkage toward the global delta baseline is tuned on the validation fold

After all folds are complete, the notebook performs a global alpha sweep on the out-of-fold predictions and stores:
- `ALPHA_SHRINK`
- `MEDIAN_EPOCHS`

These values are reused in the final full-data refit.

In [ ]:
def apply_shrink(pred, baseline, alpha):
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def aug_weight_for_epoch(epoch, warmup=AUG_WARMUP_EPOCHS, ramp=AUG_RAMP_EPOCHS, pmax=AUG_P_MAX):
    epoch = int(epoch)
    warmup = int(warmup)
    ramp = int(ramp)
    pmax = float(pmax)

    if epoch <= warmup:
        return 0.0
    if ramp <= 0:
        return pmax

    t = (epoch - warmup) / float(ramp)
    t = max(0.0, min(1.0, t))
    return float(pmax) * t

def train_one_fold(tr_idx, va_idx, fold_art, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    Z_ref_t = torch.tensor(fold_art["Z_ref"], device=device, dtype=torch.float32)
    Y_ref_t = torch.tensor(fold_art["Y_ref"], device=device, dtype=torch.float32)
    Z_va_t = torch.tensor(fold_art["Z_query"], device=device, dtype=torch.float32)
    BASIS_INIT_t = torch.tensor(fold_art["BASIS_INIT"], device=device, dtype=torch.float32)
    GENE_BIAS_INIT_t = torch.tensor(fold_art["GENE_BIAS_INIT"], device=device, dtype=torch.float32)

    model = PlainBasisHyperNet(
        d_pert=fold_art["d_pert"],
        basis_init=BASIS_INIT_t,
        hidden=HNET_HIDDEN,
        dropout=HNET_DROPOUT,
    ).to(device)
    model.set_gene_bias_init(GENE_BIAS_INIT_t)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx, dtype=np.int64)
    va_idx = np.asarray(va_idx, dtype=np.int64)

    aug_enabled = (
        AUGMENT_H5AD
        and (boot_raw is not None)
        and (h5_mean_raw is not None)
        and (boot_ok is not None)
    )

    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if aug_enabled:
        ok_mask = (boot_ok[tr_idx] == 1)
        tr_fit = tr_idx[ok_mask]

        if len(tr_fit) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[tr_fit].astype(np.float32)
            B = D_train[tr_fit].astype(np.float32)

            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(
                slopes,
                float(AUG_SLOPE_CLAMP_MIN),
                float(AUG_SLOPE_CLAMP_MAX),
            ).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32)
            rng_aug = np.random.RandomState(seed + 2027)
        else:
            aug_enabled = False

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_va_pred = None
    best_mean_term = 0.0
    best_wcos = 0.0
    patience = 0

    n_tr = len(tr_idx)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm_local = np.arange(n_tr, dtype=np.int64)
        np.random.shuffle(perm_local)

        for start in range(0, n_tr, BATCH_GENES):
            b_local = perm_local[start:start + BATCH_GENES]
            b_global = tr_idx[b_local]

            b_local_t = torch.tensor(b_local, device=device, dtype=torch.long)
            b_global_t = torch.tensor(b_global, device=device, dtype=torch.long)

            pred, _ = model.forward_parts(Z_ref_t.index_select(0, b_local_t))
            dt_b = Y_ref_t.index_select(0, b_local_t)

            aug_p_now = aug_weight_for_epoch(epoch)

            if aug_enabled and (slopes_t is not None) and (aug_p_now > 0.0):
                covered_mask_np = (boot_ok[b_global] == 1)

                if np.any(covered_mask_np):
                    kk = rng_aug.randint(0, BOOT_K, size=len(b_global))
                    dt_raw_np = boot_raw[b_global, kk, :]
                    dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                    dt_aug = dt_raw_t * slopes_t + intercepts_t

                    covered_mask_t = torch.tensor(
                        covered_mask_np[:, None],
                        device=device,
                        dtype=torch.bool,
                    )

                    dt_mix = (1.0 - float(aug_p_now)) * dt_b + float(aug_p_now) * dt_aug
                    dt_b = torch.where(covered_mask_t, dt_mix, dt_b)

            bw_b = baseline_wmae_t.index_select(0, b_global_t)

            loss_main = weighted_l1_like_rowweighted(
                dt_b,
                pred,
                bw_b,
                clamp_min=0.5,
                clamp_max=3.0,
            )
            loss_basis_drift = LAMBDA_BASIS_DRIFT * torch.mean(model.basis_delta.square())
            loss_basis_gain = LAMBDA_BASIS_GAIN * torch.mean((model.basis_gain - 1.0) ** 2)
            loss = loss_main + loss_basis_drift + loss_basis_gain

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Z_va_t).detach().cpu().numpy().astype(np.float32)

            va_true = D_train[va_idx].astype(np.float32)

            sc_best = -1e18
            a_best = 0.0
            best_dbg = None

            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                out = score_delta(va_true, pred_a, va_idx)
                sc = out["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)
                    best_dbg = out

            if sc_best > best_score:
                best_score = float(sc_best)
                best_alpha = float(a_best)
                best_epoch = int(epoch)
                best_va_pred = va_pred.copy()
                if best_dbg is not None:
                    best_mean_term = float(best_dbg.get("mean_term", 0.0))
                    best_wcos = float(best_dbg.get("wcos", 0.0))
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_va_pred, best_mean_term, best_wcos

N = D_train.shape[0]
groups = np.asarray([str(x).strip().upper() for x in train_genes])
unique_groups = np.unique(groups)

N_SPLITS = min(8, len(unique_groups))
gkf = GroupKFold(n_splits=N_SPLITS)

print(f"[split] total rows={N} | unique perts={len(unique_groups)} | n_splits={N_SPLITS}")

oof_pred = np.zeros_like(D_train, dtype=np.float32)

fold_scores = []
fold_epochs = []
fold_alphas = []
fold_mean_terms = []
fold_wcos = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(np.arange(N), groups=groups), 1):
    va_groups = set(groups[va_idx])

    va_genes = [str(train_genes[i]).strip().upper() for i in va_idx]
    fold_art = build_fold_artifacts(tr_idx, query_genes=va_genes)

    print(
        f"[fold {fold}] d_pert={fold_art['d_pert']} "
        f"basis_k={fold_art['basis_k_eff']} "
        f"svd_var={fold_art['svd_var_sum']:.4f} "
        f"basis_var={fold_art['basis_var_sum']:.4f} "
        f"ctrl_ref_cov={fold_art['ctrl_ref_cov']} "
        f"ctrl_query_cov={fold_art['ctrl_query_cov']}"
    )

    seed_scores = []
    seed_alphas = []
    seed_epochs = []
    seed_va_preds = []
    seed_mean_terms = []
    seed_wcos = []

    for s in MODEL_SEEDS:
        (
            best_score,
            best_alpha,
            best_epoch,
            best_va_pred,
            best_mean_term,
            best_wcos_,
        ) = train_one_fold(tr_idx, va_idx, fold_art, seed=int(s))

        seed_scores.append(float(best_score))
        seed_alphas.append(float(best_alpha))
        seed_epochs.append(int(best_epoch))
        seed_va_preds.append(best_va_pred.astype(np.float32, copy=False))
        seed_mean_terms.append(float(best_mean_term))
        seed_wcos.append(float(best_wcos_))

    va_pred_mean = np.mean(np.stack(seed_va_preds, axis=0), axis=0).astype(np.float32)
    oof_pred[va_idx] = va_pred_mean

    fold_factor = float(N / len(va_idx))
    seed_scores_scaled = [sc * fold_factor for sc in seed_scores]
    fold_score_scaled = float(np.mean(seed_scores_scaled))

    fold_scores.append(fold_score_scaled)
    fold_alphas.append(float(np.mean(seed_alphas)))
    fold_epochs.append(int(np.median(seed_epochs)))
    fold_mean_terms.append(float(np.mean(seed_mean_terms)))
    fold_wcos.append(float(np.mean(seed_wcos)))

    seeds_str = ", ".join([f"{sc:.6f}" for sc in seed_scores_scaled])

    print(
        f"fold {fold}: score_mean={fold_score_scaled:.6f} "
        f"scores=[{seeds_str}] "
        f"alpha_mean={np.mean(seed_alphas):.3f} "
        f"epoch_median={int(np.median(seed_epochs))} "
        f"mean_term_mean={np.mean(seed_mean_terms):.5f} "
        f"wcos_mean={np.mean(seed_wcos):.5f} "
        f"val_n={len(va_idx)} "
        f"val_perts={len(va_groups)}"
    )

print("group-cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
print("median best_epoch =", int(np.median(fold_epochs)))
print("mean_term overall:", float(np.mean(fold_mean_terms)), "wcos overall:", float(np.mean(fold_wcos)))

best_global_alpha = 0.0
best_global_score = -1e18
all_idx = np.arange(N, dtype=np.int64)

for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(D_train, pred_a, all_idx)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)

ALPHA_SHRINK = best_global_alpha
MEDIAN_EPOCHS = int(np.median(fold_epochs))

fold 1: score_mean=6.222320 scores=[6.432640, 6.093840, 6.140480] alpha_mean=0.744 epoch_median=65 mean_term_mean=0.23034 wcos_mean=0.33766  
fold 2: score_mean=6.436053 scores=[6.352800, 6.628160, 6.327200] alpha_mean=0.567 epoch_median=90 mean_term_mean=0.24143 wcos_mean=0.33323  
fold 3: score_mean=6.869440 scores=[7.068480, 6.781120, 6.758720] alpha_mean=0.723 epoch_median=75 mean_term_mean=0.24375 wcos_mean=0.35226  
fold 4: score_mean=5.404427 scores=[5.502640, 5.356240, 5.354400] alpha_mean=0.841 epoch_median=70 mean_term_mean=0.22000 wcos_mean=0.30709  
fold 5: score_mean=5.660533 scores=[5.593760, 5.561840, 5.826000] alpha_mean=0.656 epoch_median=100 mean_term_mean=0.23684 wcos_mean=0.29873  
fold 6: score_mean=7.873787 scores=[7.736560, 8.034720, 7.850080] alpha_mean=0.747 epoch_median=55 mean_term_mean=0.25255 wcos_mean=0.38969  
fold 7: score_mean=7.197440 scores=[7.185680, 7.292720, 7.113920] alpha_mean=0.693 epoch_median=125 mean_term_mean=0.21960 wcos_mean=0.40971  
fold 8: score_mean=5.297760 scores=[5.282000, 5.502480, 5.108800] alpha_mean=0.734 epoch_median=120 mean_term_mean=0.21915 wcos_mean=0.30212  
group-cv mean: 6.37022 std: 0.8519842336308555  
median best_epoch = 82  
mean_term overall: 0.23295697794652911 wcos overall: 0.34130945504951066  
OOF global alpha: 0.7040404081344604 OOF score: 6.16979  

## 12. Full refit and submission creation

This final section trains the model on all 80 training perturbations and predicts all released perturbations.

The workflow is:
1. build full-data artifacts
2. train one refit model per seed for `MEDIAN_EPOCHS`
3. average the predictions across seeds
4. apply the tuned shrinkage factor
5. write the final submission CSV in sample-submission format

Rows without a resolved released perturbation fallback to the global delta baseline.

In [ ]:
pert_all = pd.read_csv(PERT_IDS_ALL_CSV).copy()
pert_all.columns = [str(c).strip().lower() for c in pert_all.columns]

pert_all["pert_id"] = pert_all["pert_id"].astype(str)
pert_all["pert"] = pert_all["pert"].astype(str).str.strip().str.upper()

released_pid_to_gene = dict(zip(pert_all["pert_id"], pert_all["pert"]))
released_genes_unique = list(dict.fromkeys(pert_all["pert"].tolist()))

print(
    f"[released] rows={len(pert_all)} "
    f"unique_pert_ids={pert_all['pert_id'].nunique()} "
    f"unique_perts={len(released_genes_unique)}"
)

all_train_idx = np.arange(len(train_genes), dtype=np.int64)

full_art = build_fold_artifacts(
    tr_idx=all_train_idx,
    query_genes=released_genes_unique,
)

Y_full = full_art["Y_ref"].astype(np.float32)
Y_full_t = torch.tensor(Y_full, device=device, dtype=torch.float32)
Z_full_t = torch.tensor(full_art["Z_ref"], device=device, dtype=torch.float32)
Z_release_t = torch.tensor(full_art["Z_query"], device=device, dtype=torch.float32)
BASIS_INIT_full_t = torch.tensor(full_art["BASIS_INIT"], device=device, dtype=torch.float32)
GENE_BIAS_INIT_full_t = torch.tensor(full_art["GENE_BIAS_INIT"], device=device, dtype=torch.float32)

N_full = Y_full.shape[0]

release_gene_to_idx = {
    str(g).strip().upper(): i
    for i, g in enumerate(full_art["query_genes"])
}

print(
    f"[full_art] d_pert={full_art['d_pert']} "
    f"basis_k={full_art['basis_k_eff']} "
    f"svd_var={full_art['svd_var_sum']:.4f} "
    f"basis_var={full_art['basis_var_sum']:.4f} "
    f"ctrl_ref_cov={full_art['ctrl_ref_cov']} "
    f"ctrl_query_cov={full_art['ctrl_query_cov']}"
)

def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = PlainBasisHyperNet(
        d_pert=full_art["d_pert"],
        basis_init=BASIS_INIT_full_t,
        hidden=HNET_HIDDEN,
        dropout=HNET_DROPOUT,
    ).to(device)
    model.set_gene_bias_init(GENE_BIAS_INIT_full_t)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    aug_enabled = False
    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if AUGMENT_H5AD:
        assert (boot_raw is not None) and (h5_mean_raw is not None) and (boot_ok is not None), (
            "AUGMENT_H5AD=True but boot_raw/h5_mean_raw/boot_ok not built."
        )

        fit_idx = np.where(np.asarray(boot_ok).reshape(-1) == 1)[0]

        if len(fit_idx) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[fit_idx].astype(np.float32)
            B = Y_full[fit_idx].astype(np.float32)

            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(
                slopes,
                float(AUG_SLOPE_CLAMP_MIN),
                float(AUG_SLOPE_CLAMP_MAX),
            ).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32)
            rng_aug = np.random.RandomState(seed + 2027)
            aug_enabled = True

    epochs_run = int(epochs_fixed) if epochs_fixed is not None else int(MEDIAN_EPOCHS)
    boot_ok_arr = np.asarray(boot_ok).reshape(-1) if boot_ok is not None else None

    for epoch in range(1, epochs_run + 1):
        model.train()
        perm = np.arange(N_full, dtype=np.int64)
        np.random.shuffle(perm)

        for start in range(0, N_full, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred, _ = model.forward_parts(Z_full_t.index_select(0, b_t))
            dt_b = Y_full_t.index_select(0, b_t)

            aug_p_now = aug_weight_for_epoch(epoch)

            if aug_enabled and (slopes_t is not None) and (aug_p_now > 0.0):
                covered_mask_np = (boot_ok_arr[b] == 1)

                if np.any(covered_mask_np):
                    kk = rng_aug.randint(0, BOOT_K, size=len(b))
                    dt_raw_np = boot_raw[b, kk, :]
                    dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                    dt_aug = dt_raw_t * slopes_t + intercepts_t

                    covered_mask_t = torch.tensor(
                        covered_mask_np[:, None],
                        device=device,
                        dtype=torch.bool,
                    )

                    dt_mix = (1.0 - float(aug_p_now)) * dt_b + float(aug_p_now) * dt_aug
                    dt_b = torch.where(covered_mask_t, dt_mix, dt_b)

            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss_main = weighted_l1_like_rowweighted(
                dt_b,
                pred,
                bw_b,
                clamp_min=0.5,
                clamp_max=3.0,
            )
            loss_basis_drift = LAMBDA_BASIS_DRIFT * torch.mean(model.basis_delta.square())
            loss_basis_gain = LAMBDA_BASIS_GAIN * torch.mean((model.basis_gain - 1.0) ** 2)
            loss = loss_main + loss_basis_drift + loss_basis_gain

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

    model.eval()
    return model

models = [fit_full_model(int(sd), epochs_fixed=int(MEDIAN_EPOCHS)) for sd in MODEL_SEEDS]
print("Refit models:", len(models))

with torch.no_grad():
    pred_release_models = []
    for model in models:
        y = model(Z_release_t).detach().cpu().numpy().astype(np.float32)
        y = apply_shrink(y, delta_baseline, float(ALPHA_SHRINK)).astype(np.float32)
        pred_release_models.append(y)

pred_release_mean = np.mean(np.stack(pred_release_models, axis=0), axis=0).astype(np.float32)

sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)

sub_gene_cols = [c for c in sub.columns if c != "pert_id"]
idx_map = {str(g).upper(): i for i, g in enumerate(gene_columns)}
perm = [idx_map[str(g).upper()] for g in sub_gene_cols]

sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

hit = 0
miss_pid = []
miss_gene = []

for pid in sub["pert_id"].tolist():
    gene = released_pid_to_gene.get(str(pid))
    if gene is None:
        miss_pid.append(str(pid))
        continue

    q_idx = release_gene_to_idx.get(str(gene).strip().upper())
    if q_idx is None:
        miss_gene.append(gene)
        continue

    vec = pred_release_mean[q_idx, perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "ZETRO4.csv"
sub.to_csv(out_path, index=False)

print("[ok] wrote:", out_path)
print("[sub] filled rows:", hit, "/", len(sub))
print("[sub] missing pert_ids:", len(miss_pid))
if miss_pid:
    print(miss_pid[:20])

print("[sub] missing genes:", len(set(miss_gene)))
if miss_gene:
    print(sorted(set(miss_gene))[:20])

[released] rows=120 unique_pert_ids=120 unique_perts=120
[full_art] d_pert=79 basis_k=32 svd_var=0.9978 basis_var=0.8313 ctrl_ref_cov=80 ctrl_query_cov=120
[pert emb mode] mode=ctrl_all ctrl_blend_alpha=0.50
[released source counts] {'ctrl': 120}
Refit models: 11
[ok] wrote: ZETRO4.csv
[sub] filled rows: 120 / 120
[sub] missing pert_ids: 0
[sub] missing genes: 0
